## Things to improve:

- Make pass to more centrish and open player when opp_prox to defense is high
- Defensive strategy for wingLaunch tactic
- Try to train a better multi class classifier
- ~Dont slide near or inside box~
- Clear ball to side if opp_prox too high in the front
- ~Dont stop sprint in attacking area if player in close pursuit~

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use('ggplot')
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [ ]:
df = pd.read_csv('../input/offense-dist-heading/plays_head_dist.csv')
df.head()

In [ ]:
df.shape

In [ ]:
X = df.drop(['action'],axis=1).values

df.action = df.action.astype(int)

y = df['action'].values

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.02,random_state=42)

In [ ]:
# # #Setup a classifier

# clf = RandomForestClassifier()
# model=clf.fit(X_train,y_train)
# clf.score(X_test,y_test)

In [ ]:
# from xgboost import XGBClassifier

# xgb = XGBClassifier(random_state=101, max_depth= 3, learning_rate = 0.01, n_estimators = 1000, n_jobs=-1)
# model=xgb.fit(X_train,y_train)
# xgb.score(X_test,y_test)

In [ ]:
#Setup a classifier
clf = RandomForestClassifier()
model=clf.fit(X,y)

In [ ]:
# Install:
# Kaggle environments.
#!git clone https://github.com/Kaggle/kaggle-environments.git
#!cd kaggle-environments && pip install .

# GFootball environment.
!apt-get update -y
!apt-get install -y libsdl2-gfx-dev libsdl2-ttf-dev

# Make sure that the Branch in git clone and in wget call matches !!
!git clone -b v2.6 https://github.com/google-research/football.git
!mkdir -p football/third_party/gfootball_engine/lib

!wget https://storage.googleapis.com/gfootball/prebuilt_gameplayfootball_v2.6.so -O football/third_party/gfootball_engine/lib/prebuilt_gameplayfootball.so
!cd football && GFOOTBALL_USE_PREBUILT_SO=1 pip3 install .

!mkdir /kaggle_simulations
!mkdir /kaggle_simulations/agent
!mkdir /kaggle_simulations/agent/saved_model

In [ ]:
# save the model to disk
# import os
# if not os.path.exists('/kaggle_simulations/agent/RLapprox/'):
#     os.makedirs('/kaggle_simulations/agent/RLapprox/')

import pickle
filename = '/kaggle_simulations/agent/saved_model/model.sav'
# filename = '/kaggle/working/RLapprox/rfmodel.sav'
pickle.dump(model, open(filename, 'wb'))

# load the model from disk
#loaded_model = pickle.load(open(filename, 'rb'))
#result = loaded_model.score(X_test, y_test)
#print(result)

In [ ]:
%%writefile /kaggle_simulations/agent/main.py
from kaggle_environments.envs.football.helpers import *
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import pickle
import math
from random import randint

# load the model from disk
# filename = '/kaggle/working/RLapprox/rfmodel.sav'
filename = '/kaggle_simulations/agent/saved_model/model.sav'
loaded_model = pickle.load(open(filename, 'rb'))
#result = loaded_model.score(X_test, y_test)
#print(result)

directions = [[Action.TopLeft, Action.Top, Action.TopRight],
[Action.Left, Action.Idle, Action.Right],
[Action.BottomLeft, Action.Bottom, Action.BottomRight]]

#track raw data to troubleshoot...
track_raw_data=[]

perfectRange = [[0.7, 0.95], [-0.12, 0.12]]

def inside(pos, area):
    return area[0][0] <= pos[0] <= area[0][1] and area[1][0] <= pos[1] <= area[1][1]

def get_distance(pos1,pos2):
    return ((pos1[0]-pos2[0])**2+(pos1[1]-pos2[1])**2)**0.5

def get_heading(pos1,pos2):
    raw_head=math.atan2(pos1[0]-pos2[0],pos1[1]-pos2[1])/math.pi*180

    if raw_head<0:
        head=360+raw_head
    else:
        head=raw_head
    return head


def get_action(action_num):
    if action_num==0:
        return Action.Idle
    if action_num==1:
        return Action.Left
    if action_num==2:
        return Action.TopLeft
    if action_num==3:
        return Action.Top
    if action_num==4:
        return Action.TopRight
    if action_num==5:
        return Action.Right
    if action_num==6:
        return Action.BottomRight
    if action_num==7:
        return Action.Bottom
    if action_num==8:
        return Action.BottomLeft
    if action_num==9:
        return Action.LongPass
    if action_num==10:
        return Action.HighPass
    if action_num==11:
        return Action.ShortPass
    if action_num==12:
        return Action.Shot
    if action_num==13:
        return Action.Sprint
    if action_num==14:
        return Action.ReleaseDirection
    if action_num==15:
        return Action.ReleaseSprint
    if action_num==16:
        #return Action.Sliding
        return Action.Idle
    if action_num==17:
        return Action.Dribble
    if action_num==18:
        #return Action.ReleaseDribble
        return Action.Idle
    return Action.Right

# Function to update snapShot
def set_snapShot(num):
    global snapShot
    snapShot = num
    return

# Function to get snapShot
def get_snapShot():
    global snapShot
    return snapShot


# Set game plan parameters
goalRange = 0.75
crossAngleRange = 0.9
wingRange = 0.2
wingPassRange = 0.12
wingBackRange = -0.15
snapShot = 0


@human_readable_agent
def agent(obs):
    controlled_player_pos = obs['left_team'][obs['active']]
    x = controlled_player_pos[0]
    y = controlled_player_pos[1]
    pactive=obs['active']

    # Add direction to action
    def sticky_check(action, direction):
        if direction in obs['sticky_actions']:
            return action
        else:
            return direction

    # Sprint to dribble
    def sprint2dribble():
        if Action.Sprint in obs['sticky_actions']:
            return Action.ReleaseSprint
        else:
            return Action.Dribble

    # Dribble to sprint
    def dribble2sprint():
        if Action.Dribble in obs['sticky_actions']:
            return Action.ReleaseDribble
        else:
            return Action.Sprint

    # Get coordinate distance
    def get_distance_4xy(x1, y1, x2, y2):
    #     """ get two-dimensional Euclidean distance, considering y size of the field """
        return math.sqrt((x1 - x2) ** 2 + (y1 * 2.38 - y2 * 2.38) ** 2)

    # Check if player open
    def opp_prox(x, y, prox=0.05, front=False, back=False):
        xr = x + prox
        xl = x - prox
        yt = y - prox
        yb = y + prox

        if (x+prox) > 1:
            xr = 1
        if (x-prox) < -1:
            xl = -1
        if (y-prox) < -0.42:
            yt = -0.42
        if (y+prox) > 0.42:
            yb = 0.42

        cnt=0
        for opp in obs['right_team']:
            # If only checking opponents to the front
            if front:
                if opp[0] > x and opp[0] < xr and opp[1] > yt and opp[1] < yb:
                    cnt+=1
                    continue
            if back:
                if opp[0] < x and opp[0] > xl and opp[1] > yt and opp[1] < yb:
                    cnt+=1
                    continue
            # If checking in all directions
            if opp[0] > xl and opp[0] < xr and opp[1] > yt and opp[1] < yb:
                cnt+=1

        return cnt


    # Logical checks

    if obs['ball_owned_team'] == 1 or abs(controlled_player_pos[1]) > wingRange:
        set_snapShot(0)

    if get_snapShot() == 1:
        return Action.Shot

    ###############
    # Game modes
    ###############

    # Pass when KickOff or FreeKick or ThrowIn
    if obs['game_mode'] == GameMode.KickOff or obs['game_mode'] == GameMode.ThrowIn:
        return sticky_check(Action.ShortPass, Action.Right)

    # Shoot when freekick in goal range; If on wing then cross; Otherwise just pass
    if obs['game_mode'] == GameMode.FreeKick:
        # Shoot if in range
        if controlled_player_pos[0] > goalRange and abs(controlled_player_pos[1]) < wingRange:
            set_snapShot(1)
            return Action.Shot
        # Cross from right
        if controlled_player_pos[0] > goalRange and controlled_player_pos[1] > wingRange:
            if controlled_player_pos[0] > crossAngleRange:
                set_snapShot(1)
                return sticky_check(Action.HighPass, Action.Top)
            else:
                set_snapShot(1)
                return sticky_check(Action.HighPass, Action.TopRight)
        # Cross from left
        if controlled_player_pos[0] > goalRange and controlled_player_pos[1] < -(wingRange):
            if controlled_player_pos[0] > crossAngleRange:
                set_snapShot(1)
                return sticky_check(Action.HighPass, Action.Bottom)
            else:
                set_snapShot(1)
                return sticky_check(Action.HighPass, Action.BottomRight)

    # Cross in for corner
    if obs['game_mode'] == GameMode.Corner and obs['ball'][1] < 0:
        set_snapShot(1)
        return sticky_check(Action.HighPass, Action.Bottom)
    elif obs['game_mode'] == GameMode.Corner and obs['ball'][1] > 0:
        set_snapShot(1)
        return sticky_check(Action.HighPass, Action.Top)

    # Long pass when GoalKick
    if obs['game_mode'] == GameMode.GoalKick:
        # Pass to wingbacks if free
        if opp_prox(obs['left_team'][2][0], obs['left_team'][2][1]) == 0:
#             set_wingLaunch(1)
            return sticky_check(Action.ShortPass, Action.TopRight)
        elif opp_prox(obs['left_team'][3][0], obs['left_team'][3][1]) == 0:
#             set_wingLaunch(1)
            return sticky_check(Action.ShortPass, Action.BottomRight)
        # Launch downfield
        ydir = randint(0,2)
        return sticky_check(Action.LongPass, directions[ydir][2])

    # Shoot when Penalty
    if obs['game_mode'] == GameMode.Penalty:
        set_snapShot(1)
        return Action.Shot


    # Make sure player is running.
    if  controlled_player_pos[0] < 0.6 and Action.Sprint not in obs['sticky_actions']:
        return dribble2sprint() 
    elif 0.6 < controlled_player_pos[0] and Action.Sprint in obs['sticky_actions']:
        if opp_prox(controlled_player_pos[0], controlled_player_pos[1], prox=0.02, back=True) == 0:
            return sprint2dribble()
        else:
            return dribble2sprint()

    #if we have the ball
    if obs['ball_owned_player'] == obs['active'] and obs['ball_owned_team'] == 0:

        # Defensive position
        # Clear if we are near our goal
        if controlled_player_pos[0] < -(goalRange) and abs(controlled_player_pos[1]) < wingPassRange:
            return Action.Shot

        # Wingback region
        if controlled_player_pos[0] < wingBackRange and abs(controlled_player_pos[1]) > wingPassRange:
            if opp_prox(controlled_player_pos[0], controlled_player_pos[1], front=True) == 0:
                # Launch down wing
                return sticky_check(Action.HighPass, Action.Right)
            else:
                return Action.Shot

        # Offensive positions
        # Shoot if we are in the final third and not at an acute angle
        if controlled_player_pos[0] > goalRange and abs(controlled_player_pos[1]) < wingRange:
            set_snapShot(1)
            return Action.Shot
        # Shoot if goalie out of position
        elif (obs['right_team'][0][0] < 0.83 or abs(obs['right_team'][0][1]) > 0.05) and (controlled_player_pos[0] > 0.3) and abs(controlled_player_pos[1]) < wingRange:
            set_snapShot(1)
            return Action.Shot
        # Shoot if attacker very close to goalie
        # Player one on one with goalie
        goalkeeper_x = obs["right_team"][0][0] + obs["right_team_direction"][0][0] * 10
        goalkeeper_y = obs["right_team"][0][1] + obs["right_team_direction"][0][1] * 10
        # player have the ball and located close to the goalkeeper
        if get_distance_4xy(controlled_player_pos[0], controlled_player_pos[1], goalkeeper_x, goalkeeper_y) < 0.25:
            set_snapShot(1)
            return Action.Shot

        # Crossing positions
        # Cross from right
        if controlled_player_pos[0] > goalRange and controlled_player_pos[1] > wingRange:
            if controlled_player_pos[0] > crossAngleRange:
                set_snapShot(1)
                return sticky_check(Action.HighPass, Action.Top)
            else:
                set_snapShot(1)
                return sticky_check(Action.HighPass, Action.TopRight)

        # Cross from left
        if controlled_player_pos[0] > goalRange and controlled_player_pos[1] < -(wingRange):
            if controlled_player_pos[0] > crossAngleRange:
                set_snapShot(1)
                return sticky_check(Action.HighPass, Action.Bottom)
            else:
                set_snapShot(1)
                return sticky_check(Action.HighPass, Action.BottomRight)

        dat=[]
        to_append=[]
        #return Action.Right
        #get controller player pos
        controlled_player_pos = obs['left_team'][obs['active']]

        if inside(controlled_player_pos, perfectRange) and controlled_player_pos[0] < obs['ball'][0]:
            return Action.Shot

        goalx=0.0
        goaly=0.0

        sidelinex=0.0
        sideliney=0.42

        goal_dist=get_distance((x,y),(goalx,goaly))
        sideline_dist=get_distance((x,y),(sidelinex,sideliney))
        to_append.append(goal_dist)
        to_append.append(sideline_dist)

        for i in range(len(obs['left_team'])):
            dist=get_distance((x,y),(obs['left_team'][i][0],obs['left_team'][i][1]))
            head=get_heading((x,y),(obs['left_team'][i][0],obs['left_team'][i][1]))
            to_append.append(dist)
            to_append.append(head)

        for i in range(len(obs['right_team'])):
            dist=get_distance((x,y),(obs['right_team'][i][0],obs['right_team'][i][1]))
            head=get_heading((x,y),(obs['right_team'][i][0],obs['right_team'][i][1]))
            to_append.append(dist)
            to_append.append(head)

        dat.append(to_append)

        predicted=loaded_model.predict(dat)

        do=get_action(predicted)

        if do == None:
            return Action.Right
        else:
            return do

    # if we don't have ball run to ball
    else:
        controlled_player_pos = obs['left_team'][obs['active']]
        dirsign = lambda x: 1 if abs(x) < 0.01 else (0 if x < 0 else 2)
        #where ball is going
        ball_targetx=obs['ball'][0]+(obs['ball_direction'][0]*5)
        ball_targety=obs['ball'][1]+(obs['ball_direction'][1]*5)

        e_dist=get_distance(obs['left_team'][obs['active']],obs['ball'])

        if e_dist >.01:
            # Run where ball will be
            xdir = dirsign(ball_targetx - controlled_player_pos[0])
            ydir = dirsign(ball_targety - controlled_player_pos[1])
            return directions[ydir][xdir]
        else:
            prob = randint(0,100)
            if prob > 65 and controlled_player_pos[0] > -(goalRange-0.35):
                return Action.Slide
            # Run towards the ball.
            xdir = dirsign(obs['ball'][0] - controlled_player_pos[0])
            ydir = dirsign(obs['ball'][1] - controlled_player_pos[1])
            return directions[ydir][xdir]

In [ ]:
# Set up the Environment.
from kaggle_environments import make
env = make("football", debug=True, configuration={"save_video": True, "scenario_name": "11_vs_11_kaggle", "running_in_notebook": True})
output = env.run(["/kaggle_simulations/agent/main.py", "do_nothing"])[-1]
print('Left player: reward = %s, status = %s, info = %s' % (output[0]['reward'], output[0]['status'], output[0]['info']))
print('Right player: reward = %s, status = %s, info = %s' % (output[1]['reward'], output[1]['status'], output[1]['info']))
env.render(mode="human", width=800, height=600)

In [ ]:
# import tarfile
# tar = tarfile.open("submit.tar.gz", "w:gz")
# tar.add("submission.py")
# tar.addfile(tarfile.TarInfo("RLapprox/rfmodel.sav"), open("/kaggle/working/RLapprox/rfmodel.sav"))
# tar.close()

In [ ]:
!cd /kaggle_simulations/agent && tar -czvf /kaggle/working/submit.tar.gz main.py saved_model